# Publication notes
See [the project README](../README.md) and [setup instructions](../docs/setup.md). Outputs were cleared for publication; this file has not been rerun during packaging. Notebook 6 is the current integration development reference; supplied-run evidence and limitations are documented in the README.


## 1. Locate the existing repositories and load Drake


In [ ]:
import os
import ast
import importlib.util
import json
import sys
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, LinearConstraint

PROJECT = Path(os.environ.get("PUSHER_PROJECT", Path.cwd().parent)).expanduser().resolve()
CONTACT_REPO = Path(os.environ.get("CONTACT_SLS_REPO", Path.home() / "projects/quasistatic-contact-SLS")).expanduser().resolve()
ADAPTIVE_REPO = Path(os.environ.get("ADAPTIVE_SLS_REPO", Path.home() / "projects/adaptive_sls_complete")).expanduser().resolve()
ALGORITHM_DIR = CONTACT_REPO / "examples/planning_through_contact/algorithms"

try:
    from pydrake.all import QuasidynamicLinearPusher
except ImportError as exc:
    raise RuntimeError("Select the same working Drake kernel used for Notebook 1.") from exc

def load_source_module(name, path):
    if not path.is_file():
        raise FileNotFoundError(f"Expected source file: {path}")
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


TrajOptProblem = load_source_module(
    "probe5_traj_opt_problem", ALGORITHM_DIR / "traj_opt_problem.py"
).TrajOptProblem
IterativeTO = load_source_module(
    "probe5_iterative_to", ALGORITHM_DIR / "iterative_to.py"
).IterativeTO

print("Python:", sys.executable)
print("Drake planning source:", ALGORITHM_DIR)
print("Adaptive source:", ADAPTIVE_REPO / "src/gpu_sls/gpu_sls.py")
print("Backend: original Drake IterativeTO; no automatic substitute backend.")


## 2. Declare the prior, observation assumptions, and experiment limits


In [ ]:
K_A, H, W = 1.0, 0.02, 0.1
N = 6
MASS_PRIOR = np.array([0.005, 0.030])  # kg
POSITION_NOISE = 5e-6                  # absolute position error, m
OBJECT_ALLOWANCE = 5e-6                # empirical one-step object discrepancy, m
PUSHER_ALLOWANCE = 120e-6              # empirical one-step pusher discrepancy, m
MIN_EXCITATION = 1e-3                  # ensures active contact throughout the prior
MAX_TRAVEL = 8e-3                      # relative to the actual initial object position
MAX_FORCE = 0.025                      # N, including numerical pusher allowance
MAX_COMMAND_STEP = 12e-3               # m per sample
MAX_COMMAND_OFFSET = 35e-3             # m from measured initial contact boundary

# Normalization and weights, not claimed to be physical energy units.
MOVE_SCALE, FORCE_SCALE, COMMAND_SCALE = 2e-3, 0.020, 10e-3
MOVE_WEIGHT, FORCE_WEIGHT, COMMAND_WEIGHT = 1.0, 0.25, 0.05
INFO_WEIGHTS = [0.0, 3.0, 30.0, 300.0, 3000.0]
OUTER_ITERS = 15
FEAS_TOL = 2e-8                        # m for the normalized constraint residuals
GAIN_REG = 1e-7                        # matches regularization in normalized coordinates

def alpha_from_mass(mass):
    return K_A * H**2 / (np.asarray(mass) + K_A * H**2)

def mass_from_alpha(alpha):
    return K_A * H**2 * (1.0 / np.asarray(alpha) - 1.0)

ALPHA_LO, ALPHA_HI = alpha_from_mass(MASS_PRIOR[::-1])
ALPHA_CENTER = (ALPHA_LO + ALPHA_HI) / 2
ALPHA_RADIUS = (ALPHA_HI - ALPHA_LO) / 2
NOMINAL_MASS = float(mass_from_alpha(ALPHA_CENTER))
RESIDUAL_BOUND = (2.0 - ALPHA_LO) * POSITION_NOISE + OBJECT_ALLOWANCE

print("Mass prior [g]:", MASS_PRIOR * 1e3)
print("Planning midpoint is in alpha-space; nominal mass [g]:", NOMINAL_MASS * 1e3)
print("Duration [s]:", N * H)
print("Uniform position-residual bound [um]:", RESIDUAL_BOUND * 1e6)
print("Physical velocity/friction are not included in this model.")


## 3. Exact active-contact model and conservative planning envelope


In [ ]:
def analytical_step(state, command, mass):
    p, o = np.asarray(state, dtype=float)
    a = max(float(command) + W - o, 0.0)
    onext = o + float(alpha_from_mass(mass)) * a
    pnext = min(float(command), onext - W)
    return np.array([pnext, onext])

# Define physical constrains
def planning_quantities(commands, y0):
    """Accept numeric or symbolic commands. All recursions are affine in commands."""
    lo, hi, nominal = [y0 - POSITION_NOISE], [y0 + POSITION_NOISE], [y0]
    excitation, f_upper, margins = [], [], []
    previous_command = y0 - W - POSITION_NOISE
    for u in commands:
        a = u + W - nominal[-1]
        excitation.append(a)
        gap_worst = u + W - hi[-1]
        fmax = K_A * (1 - ALPHA_LO) * (u + W - lo[-1]) + K_A * PUSHER_ALLOWANCE
        f_upper.append(fmax)
        lo.append((1 - ALPHA_LO) * lo[-1] + ALPHA_LO * (u + W) - OBJECT_ALLOWANCE)
        hi.append((1 - ALPHA_HI) * hi[-1] + ALPHA_HI * (u + W) + OBJECT_ALLOWANCE)
        nominal.append((1 - ALPHA_CENTER) * nominal[-1] + ALPHA_CENTER * (u + W))
        # All margins expressed in meters for a common feasibility tolerance.
        margins += [
            gap_worst - MIN_EXCITATION,
            MAX_TRAVEL - (hi[-1] - (y0 - POSITION_NOISE)),
            (MAX_FORCE - fmax) / K_A,
            MAX_COMMAND_STEP - (u - previous_command),
            MAX_COMMAND_STEP + (u - previous_command),
            MAX_COMMAND_OFFSET - (u - (y0 - W)),
            u - (y0 - W),
        ]
        previous_command = u
    return dict(lo=lo, hi=hi, nominal=nominal, a=excitation, force=f_upper, margins=margins)

def feasible(commands, y0):
    return np.min(np.asarray(planning_quantities(commands, y0)["margins"], float)) >= -FEAS_TOL

def disturbance_cost(commands, y0):
    q = planning_quantities(commands, y0)
    previous = y0 - W - POSITION_NOISE
    cost = 0.0
    for k, u in enumerate(commands):
        cost += MOVE_WEIGHT * ((q["hi"][k+1] - (y0-POSITION_NOISE)) / MOVE_SCALE)**2
        cost += FORCE_WEIGHT * (q["force"][k] / FORCE_SCALE)**2
        cost += COMMAND_WEIGHT * ((u - previous) / COMMAND_SCALE)**2
        previous = u
    return cost / N


## 4. Load the original simulator and check the numerical allowances


In [ ]:
class DrakePlant:
    def __init__(self, mass):
        self.system = QuasidynamicLinearPusher(
            k_a=K_A, m_o=float(mass), w=W, dt=H, dynamics_smoothing=0.0
        )
        self.context = self.system.CreateDefaultContext()

    def step(self, state, command):
        self.context.SetDiscreteState(np.asarray(state, dtype=float))
        self.system.get_input_port().FixValue(self.context, np.array([float(command)]))
        update = self.system.EvalUniquePeriodicDiscreteUpdate(self.context)
        self.context.SetDiscreteState(update)
        return self.context.get_discrete_state_vector().CopyToVector()

max_calibration = np.zeros(2)
for mass in [MASS_PRIOR[0], 0.015, MASS_PRIOR[1]]:
    plant = DrakePlant(mass)
    for o in [0.15, 0.20, 0.30]:
        for a in [0.001, 0.005, 0.010, 0.025]:
            state = np.array([o-W, o])
            u = o-W+a
            difference = np.abs(plant.step(state, u) - analytical_step(state, u, mass))
            max_calibration = np.maximum(max_calibration, difference)
print("Calibration max discrepancies [pusher, object], um:", max_calibration * 1e6)
assert max_calibration[0] <= PUSHER_ALLOWANCE, "Pusher allowance exceeded; investigate before planning."
assert max_calibration[1] <= OBJECT_ALLOWANCE, "Object allowance exceeded; investigate before planning."
print("Calibration passed on 36 active-contact cases; this is not a global error proof.")


## 5. Adapt the posterior learning equations and verify them against the source

In [ ]:
def posterior_update(generators, sensitivity, residual_bound=RESIDUAL_BOUND):
    G = np.asarray(generators, float)
    q = float(G @ G)
    radius = float(np.abs(G).sum())
    if abs(sensitivity) < 1e-10:
        return np.r_[G, 0.0], 0.0
    noise_map = residual_bound / sensitivity
    L = q / (q + noise_map**2 + GAIN_REG)
    Gplus = np.r_[(1-L)*G, L*noise_map]
    if not (abs(1-L)*radius < radius and np.abs(Gplus).sum() < radius):
        return np.r_[G, 0.0], 0.0
    return Gplus, L / sensitivity

# Read only the pure numerical function from the verified local repo.
# No JAX import, repository modification, or full adaptive controller import is needed.
source_path = ADAPTIVE_REPO / "src/gpu_sls/gpu_sls.py"
tree = ast.parse(source_path.read_text())
source_function = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef) and node.name == "adaptive_posterior_factors"
)
source_function.decorator_list = []
namespace = {"jnp": np}
exec(compile(ast.Module(body=[source_function], type_ignores=[]), str(source_path), "exec"), namespace)
reference_update = namespace["adaptive_posterior_factors"]

for label, sensitivity, b in [
    ("no sensitivity", 0.0, 0.1),
    ("informative", 1.0, 0.1),
    ("too noisy", 1.0, 2.0),
]:
    G = np.array([1.0])
    observable = abs(sensitivity) >= 1e-10
    Pi = np.array([[float(observable)]])
    Wobs = np.array([[b/sensitivity if observable else 0.0]])
    _, _, expected, _ = reference_update(G[None, :], Pi, Wobs, GAIN_REG, True)
    actual, raw_gain = posterior_update(G, sensitivity, b)
    np.testing.assert_allclose(actual, expected.ravel(), atol=1e-12)
    print(label, "radius:", np.abs(actual).sum(), "trace reward:", 1-actual@actual)
print("Scalar adapter matches the local repo's posterior algebra on these checks.")


## 6. Recompute the full forecast and construct the manual baseline


In [ ]:
def forecast(commands, y0):
    a = np.asarray(planning_quantities(commands, y0)["a"], float)
    G = np.array([1.0])
    history, gains, traces, radii = [], [], [1.0], [1.0]
    for excitation in a:
        history.append(G.copy())
        G, gain = posterior_update(G, ALPHA_RADIUS * max(excitation, 0.0))
        gains.append(gain)
        traces.append(float(G @ G))
        radii.append(float(np.abs(G).sum()))
    return dict(G=history, gains=np.array(gains), traces=np.array(traces), radii=np.array(radii))

def objective(commands, y0, weight):
    # The full-horizon undiscounted trace reward telescopes to 1 - final trace.
    return float(disturbance_cost(commands, y0) - weight*(1-forecast(commands, y0)["traces"][-1]))

def manual_commands(y0):
    # Same six requested excitations as Notebook 1, planned open loop around nominal alpha.
    o = float(y0)
    commands = []
    for a in np.array([5, 10, 15, 20, 10, 20]) * 1e-3:
        commands.append(o-W+a)
        o += ALPHA_CENTER*a
    return np.array(commands)

# Initial measurement is supplied to the planner; no true mass is passed.
Y0 = 0.20
MANUAL = manual_commands(Y0)
assert feasible(MANUAL, Y0), "Manual baseline violates configured limits; revise budgets explicitly."
print("Manual robust upper travel [mm]:", (planning_quantities(MANUAL,Y0)["hi"][-1]-(Y0-POSITION_NOISE))*1e3)
print("Manual forecast normalized radius:", forecast(MANUAL, Y0)["radii"][-1])


## 7. Optimize using Wei-Chen's `TrajOptProblem` and `IterativeTO`

In [ ]:
def frozen_cost(commands, y0, weight, frozen):
    a = planning_quantities(commands, y0)["a"]
    cost = disturbance_cost(commands, y0)
    for k in range(N):
        q = float(frozen["G"][k] @ frozen["G"][k])
        ell = float(frozen["gains"][k])
        next_q = q*(1-ell*ALPHA_RADIUS*a[k])**2 + (ell*RESIDUAL_BOUND)**2
        cost += weight*(next_q-q)
    return cost / (1.0 + weight)  # positive scaling improves QP conditioning

def solve_frozen_drake(reference, y0, weight, frozen):
    model = QuasidynamicLinearPusher(
        k_a=K_A, m_o=NOMINAL_MASS, w=W, dt=H, dynamics_smoothing=0.0
    )
    problem = TrajOptProblem(system=model, num_steps=N)
    problem.SetInitialState([y0-W-2*POSITION_NOISE, y0])
    u_symbols = [problem.input(k)[0] for k in range(N)]
    problem.AddCost(frozen_cost(u_symbols, y0, weight, frozen))
    for margin in planning_quantities(u_symbols, y0)["margins"]:
        problem.AddConstraint(margin, lb=0.0, ub=np.inf)
    trajectory = IterativeTO(
        problem, us_init=np.asarray(reference)[:, None],
        use_contact_trust_region=False,  # active contact already constrained over the prior
        use_true_dynamics_for_rollout=False,
        use_true_dynamics_for_constraint=False,
        trust_region_covar=np.eye(3) / (0.020**2),
        max_iters=8, tol=1e-9,
    )
    return np.asarray(trajectory.us, float).ravel()

def optimize_probe(y0, weight, initial, inner_solver=solve_frozen_drake):
    reference = np.asarray(initial, float).copy()
    assert feasible(reference, y0)
    history = [objective(reference, y0, weight)]
    for iteration in range(OUTER_ITERS):
        frozen = forecast(reference, y0)
        candidate = inner_solver(reference, y0, weight, frozen)
        if not np.isfinite(candidate).all():
            raise RuntimeError("Inner optimizer returned nonfinite commands.")
        accepted = False
        for scale in [1.0, 0.5, 0.25, 0.125, 0.0625, 0.03125]:
            trial = reference + scale*(candidate-reference)
            if feasible(trial, y0) and objective(trial,y0,weight) < history[-1] - 1e-10:
                change = np.max(np.abs(trial-reference))
                reference = trial
                history.append(objective(reference,y0,weight))
                accepted = True
                break
        if not accepted or change < 1e-7:
            break
    return reference, np.asarray(history)

PLANS = {"Manual": MANUAL}
OPT_HISTORY = {}
for weight in INFO_WEIGHTS:
    name = f"Optimized lambda={weight:g}"
    commands, history = optimize_probe(Y0, weight, MANUAL)
    PLANS[name], OPT_HISTORY[name] = commands, history
    q = planning_quantities(commands, Y0)
    print(f"{name:24s}  accepted updates={len(history)-1:2d}  "
          f"upper travel={1e3*(q['hi'][-1]-(Y0-POSITION_NOISE)):.3f} mm  "
          f"forecast radius={forecast(commands,Y0)['radii'][-1]:.4f}")
print("Plans fixed. Hidden plant masses have not been used for optimization.")
print("A run reaching the outer-iteration cap is not evidence of convergence; inspect OPT_HISTORY.")


## 8. Estimate mass independently from noisy positions

In [ ]:
def identify_positions(commands, measurements):
    low, high = float(ALPHA_LO), float(ALPHA_HI)
    intervals = [[low, high]]
    accepted = 0
    for k, u in enumerate(commands):
        a_y = u + W - measurements[k]
        d_y = measurements[k+1] - measurements[k]
        if a_y > POSITION_NOISE + 1e-12:
            low = max(low, (d_y-2*POSITION_NOISE-OBJECT_ALLOWANCE)/(a_y-POSITION_NOISE))
            high = min(high, (d_y+2*POSITION_NOISE+OBJECT_ALLOWANCE)/(a_y+POSITION_NOISE))
            accepted += 1
        if low > high + 1e-12:
            raise RuntimeError(f"Empty parameter interval at transition {k}; check model/noise assumptions.")
        intervals.append([low, high])
    intervals = np.asarray(intervals)
    mass_intervals = np.column_stack((mass_from_alpha(intervals[:,1]), mass_from_alpha(intervals[:,0])))
    return dict(alpha_intervals=intervals, mass_intervals=mass_intervals,
                mass_estimate=float(mass_intervals[-1].mean()), accepted=accepted)

# Check the measurement inequalities against extreme noise combinations.
from itertools import product
for m in [0.005, 0.015, 0.030]:
    alpha = float(alpha_from_mass(m))
    for a in [0.001, 0.005, 0.020]:
        for n0, n1, delta in product([-POSITION_NOISE,POSITION_NOISE],
                                     [-POSITION_NOISE,POSITION_NOISE],
                                     [-OBJECT_ALLOWANCE,OBJECT_ALLOWANCE]):
            o0 = 0.2
            data = np.array([o0+n0, o0+alpha*a+delta+n1])
            fit = identify_positions([o0-W+a], data)
            lo, hi = fit["alpha_intervals"][-1]
            assert lo-1e-12 <= alpha <= hi+1e-12
print("72 endpoint-noise/model-error corner checks passed.")


## 9. Execute the fixed plans in Drake and compare noisy observations


In [ ]:
def execute_and_evaluate(commands, mass, seed, plant_factory=DrakePlant):
    rng = np.random.default_rng(seed)
    initial_noise = rng.uniform(-POSITION_NOISE, POSITION_NOISE)
    o0 = Y0 - initial_noise          # same observed initial Y0 for every compared strategy
    x = np.array([Y0-W-2*POSITION_NOISE, o0])
    plant = plant_factory(mass)
    states, forces, discrepancies = [x.copy()], [], []
    for u in commands:
        expected = analytical_step(x, u, mass)
        next_x = plant.step(x, u)
        discrepancies.append(np.abs(next_x-expected))
        forces.append(K_A*(u-next_x[0]))
        states.append(next_x.copy())
        x = next_x
    states = np.asarray(states)
    discrepancy = np.asarray(discrepancies).max(axis=0)
    if discrepancy[0] > PUSHER_ALLOWANCE or discrepancy[1] > OBJECT_ALLOWANCE:
        raise RuntimeError(f"Numerical allowance exceeded: {discrepancy}; investigate before interpreting bounds.")
    noise = rng.uniform(-POSITION_NOISE, POSITION_NOISE, len(states))
    noise[0] = initial_noise
    measurements = states[:,1] + noise
    fit = identify_positions(commands, measurements)
    mass_intervals = fit["mass_intervals"]
    covered = bool(np.all((mass_intervals[:,0]-1e-12 <= mass) & (mass <= mass_intervals[:,1]+1e-12)))
    force = np.asarray(forces)
    travel = states[:,1] - o0
    actual_a = np.asarray(commands) + W - states[:-1,1]
    limits_ok = bool(travel.max() <= MAX_TRAVEL+FEAS_TOL and
                     force.max() <= MAX_FORCE+K_A*FEAS_TOL and
                     actual_a.min() >= MIN_EXCITATION-FEAS_TOL)
    return dict(states=states, measurements=measurements, force=force, fit=fit,
                coverage=covered, limits_ok=limits_ok, discrepancy=discrepancy,
                travel_mm=1e3*travel[-1], width_g=1e3*np.diff(mass_intervals[-1])[0],
                peak_force_mN=1e3*force.max(),
                force_squared_integral=float(H*np.sum(force**2)))

DEMO_MASS, DEMO_SEED = 0.015, 7  # evaluation only
DEMO = {name: execute_and_evaluate(u, DEMO_MASS, DEMO_SEED) for name,u in PLANS.items()}
print(f"{'Strategy':25s} {'Travel mm':>10s} {'Peak mN':>10s} {'Width g':>10s} {'Coverage':>10s} {'Limits':>8s}")
for name, result in DEMO.items():
    print(f"{name:25s} {result['travel_mm']:10.4f} {result['peak_force_mN']:10.3f} "
          f"{result['width_g']:10.4f} {str(result['coverage']):>10s} {str(result['limits_ok']):>8s}")
assert all(r["coverage"] and r["limits_ok"] for r in DEMO.values()), "Demo failed a coverage or physical-limit check."


## 10. Plot the trajectory, mass intervals, and observed tradeoff


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
times = np.arange(N+1)*H
for name, result in DEMO.items():
    axes[0].plot(times, (result["states"][:,1]-result["states"][0,1])*1e3, marker='.', label=name)
    intervals = result["fit"]["mass_intervals"]*1e3
    axes[1].plot(np.arange(N+1), intervals[:,1]-intervals[:,0], marker='.', label=name)
    axes[2].scatter(result["travel_mm"], result["width_g"], s=55, label=name)
axes[0].set(xlabel="Time [s]", ylabel="Object displacement [mm]", title="Drake execution")
axes[1].set(xlabel="Observed transitions", ylabel="Mass interval width [g]", title="Actual bounded-data identification")
axes[2].set(xlabel="Final displacement [mm]", ylabel="Final mass interval width [g]", title="Lower and left is preferable")
for ax in axes: ax.grid(alpha=.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, u in PLANS.items():
    axes[0].step(times[:-1], (u-(Y0-W))*1e3, where="post", label=name)
    axes[1].step(times[:-1], DEMO[name]["force"]*1e3, where="post", label=name)
axes[0].set(xlabel="Time [s]", ylabel="Command from initial contact boundary [mm]")
axes[1].set(xlabel="Time [s]", ylabel="Model-derived contact force [mN]")
axes[1].axhline(MAX_FORCE*1e3, color="black", linestyle="--", label="Force limit")
for ax in axes: ax.grid(alpha=.3)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


## 11. Validate across hidden masses and paired noise seeds

In [ ]:
EVALUATION_MASSES = [0.005, 0.010, 0.015, 0.020, 0.030]
EVALUATION_SEEDS = [0, 1, 2, 3, 4]
ROWS = []
for mass in EVALUATION_MASSES:
    for seed in EVALUATION_SEEDS:
        for name, commands in PLANS.items():
            r = execute_and_evaluate(commands, mass, seed)
            ROWS.append(dict(strategy=name, mass_g=mass*1e3, seed=seed,
                             width_g=r["width_g"], travel_mm=r["travel_mm"],
                             peak_force_mN=r["peak_force_mN"], coverage=r["coverage"],
                             limits_ok=r["limits_ok"], max_object_error_um=r["discrepancy"][1]*1e6))

print(f"{'Strategy':25s} {'Trials':>7s} {'Mean width g':>13s} {'Mean travel mm':>15s} {'Coverage':>10s}")
for name in PLANS:
    subset = [r for r in ROWS if r["strategy"] == name]
    print(f"{name:25s} {len(subset):7d} {np.mean([r['width_g'] for r in subset]):13.4f} "
          f"{np.mean([r['travel_mm'] for r in subset]):15.4f} "
          f"{np.mean([r['coverage'] for r in subset]):10.1%}")
print("Limit violations:", sum(not r["limits_ok"] for r in ROWS))
print("Coverage failures:", sum(not r["coverage"] for r in ROWS))

manual_rows = {(r["mass_g"],r["seed"]):r for r in ROWS if r["strategy"]=="Manual"}
print("\nPaired trials with <= manual interval width and strictly less displacement:")
for name in list(PLANS)[1:]:
    subset = [r for r in ROWS if r["strategy"]==name]
    count = sum(r["width_g"] <= manual_rows[(r["mass_g"],r["seed"])]["width_g"]+1e-9
                and r["travel_mm"] < manual_rows[(r["mass_g"],r["seed"])]["travel_mm"]-1e-6 for r in subset)
    print(name, f"{count}/{len(subset)}")
print("Zero dominance wins does not prove failure: inspect the full accuracy/disturbance tradeoff.")
print("A high coverage rate in these trials is empirical, not an unconditional guarantee.")


## 12. Save this run without overwriting earlier notebooks or results


In [ ]:
SAVE_RESULTS = True  # set True after reviewing the outputs, then rerun this cell
if SAVE_RESULTS:
    import csv
    folder = PROJECT / "results/information_aware_probing" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    folder.mkdir(parents=True, exist_ok=False)
    with (folder / "batch_results.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(ROWS[0]))
        writer.writeheader(); writer.writerows(ROWS)
    np.savez_compressed(folder / "plans.npz", names=np.array(list(PLANS)), commands=np.stack(list(PLANS.values())))
    settings = dict(K_A=K_A, H=H, W=W, N=N, mass_prior_kg=MASS_PRIOR.tolist(),
                    position_noise_m=POSITION_NOISE, object_allowance_m=OBJECT_ALLOWANCE,
                    pusher_allowance_m=PUSHER_ALLOWANCE, maximum_travel_m=MAX_TRAVEL,
                    maximum_force_N=MAX_FORCE, info_weights=INFO_WEIGHTS,
                    planner="Drake IterativeTO with frozen-generator/frozen-gain quadratic adapter",
                    contact_mode="active throughout", sensing="noisy object positions only")
    (folder / "settings.json").write_text(json.dumps(settings, indent=2))
    print("Saved:", folder)
else:
    print("Results not saved. Existing notebooks and repositories remain unchanged.")
